# From a Weather Forecast to an Electricity Forecast
## Connecticut electricity demand, one day ahead, using three weather models

**Version: Quantile Random Forest.** This notebook uses a **quantile random forest**, which gives a low, a middle and a high number per hour.
A twin notebook does the same thing with a plain random forest, if you just want one number per hour.

### What this notebook does

Electricity demand in Connecticut depends heavily on the weather. Cold mornings mean heating, hot afternoons mean air conditioning.
So if we want to know tomorrow's demand, we need tomorrow's weather. That means a **weather forecast**.

We will:

1. **Download what actually happened**: hourly Connecticut electricity demand (from ISO New England) and hourly temperature at Bradley Airport (from Iowa State's weather archive).
2. **Train a model** that learns: *"given the temperature and the time of day, how much electricity does Connecticut use?"*
3. **Pretend it is the afternoon before** a day we already know the answer to, and download what three weather models predicted at that time:
   * **HRRR** (High-Resolution Rapid Refresh)
   * **GFS** (Global Forecast System)
   * **NBM** (National Blend of Models)
4. **Feed each weather forecast into the model** and compare the predicted demand against the actual demand, and the predicted temperature against the actual temperature.

### How to run it

* In Colab: **Runtime → Run all**. It takes roughly 5 to 10 minutes. Most of that time is spent downloading weather files.
* Nothing to log into. Every data source here is public.
* To try a different day, change `target_day` in the settings cell and run everything again.

### Map of the notebook

| Part | What happens |
|---|---|
| 0. Setup | install two packages, set the day and the place |
| 1. What actually happened | electricity demand (ISO-NE) and temperature (Bradley Airport) |
| 2. The model | turn temperature + clock into features, train, check it |
| 3. The weather forecasts | HRRR, then GFS, then NBM: how each file is pulled, temperature vs actual, demand vs actual |
| 4. Side by side | all three models on one chart and one scoreboard |

---
# Part 0. Setup

Colab already has pandas, numpy, scikit-learn and matplotlib. We need two more:

* **herbie-data**: a package that finds and downloads weather-model files from NOAA's public cloud archive. It can download just the one variable we want instead of the whole file.
* **quantile-forest**: a random forest that can also give ranges (low / middle / high). The plain random forest version still installs it, it does no harm.

In [ ]:
!pip install -q herbie-data quantile-forest

In [ ]:
import io
import time
import warnings

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pandas.tseries.holiday import USFederalHolidayCalendar
from herbie import Herbie
from quantile_forest import RandomForestQuantileRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
print("ready")

## The settings

Everything you might want to change is in one place.

* `target_day`: the day we forecast. `None` means **yesterday**, so the actual answer is already known and we can grade the forecast.
* `run_hour_utc`: which run of the weather models to use. Weather models are "run" (started) at fixed times of day, written in UTC.
  `18` means the 18Z run on the day before the target day, which is **2 PM in Connecticut in summer time (1 PM in winter)**.
  That is the honest "afternoon before" forecast that a utility would actually have had.
* `train_days`: how many days of history the model learns from.
* `latitude`, `longitude`, `station`: Bradley International Airport (BDL), Windsor Locks, CT. We use one point to represent the state.
* `load_zone_id`: ISO New England's code for the Connecticut load zone (4004).

In [ ]:
SETTINGS = {
    "target_day": None,        # e.g. "2026-10-27"; None = yesterday
    "run_hour_utc": 18,        # weather model run, the day before the target day (18Z = 2 PM EDT / 1 PM EST)
    "train_days": 365,         # days of history to learn from
    "latitude": 41.9389,       # Bradley International Airport (BDL)
    "longitude": -72.6832,
    "station": "BDL",
    "load_zone_id": "4004",    # ISO-NE load zone: Connecticut
    "timezone": "America/New_York",
}
TZ = SETTINGS["timezone"]

### Turning the settings into times

Three different clocks show up in this notebook, so it's worth being careful:

* **Connecticut time** (EDT in summer, EST in winter). The target day runs from midnight to midnight Connecticut time.
* **UTC** (also written "Z"). Weather models and most data files use UTC. Connecticut is 4 hours behind UTC in summer, 5 in winter.
* **Forecast hour**, written `fxx`. A weather model run started at 18Z produces a file for 18Z + 1 hour (`fxx = 1`), 18Z + 2 hours (`fxx = 2`), and so on.
  The time a forecast is *for* is called the **valid time**: `valid time = run time + fxx hours`.

We need every forecast hour from the run time up to the last hour of the target day. The early ones (the evening before) are not graded,
but the model uses them for the "temperature over the last 24 hours" feature.

In [ ]:
if SETTINGS["target_day"] is None:
    today_ct = pd.Timestamp.now(tz=TZ).normalize()
    target_day = (today_ct.tz_localize(None) - pd.Timedelta(days=1)).tz_localize(TZ)
else:
    target_day = pd.Timestamp(SETTINGS["target_day"]).tz_localize(TZ)

# every hour of the target day in Connecticut time (23 or 25 hours on the two clock-change days)
next_day = (target_day.tz_localize(None) + pd.Timedelta(days=1)).tz_localize(TZ)
target_hours_local = pd.date_range(target_day, next_day, freq="h", inclusive="left")
target_hours_utc = target_hours_local.tz_convert("UTC")

# the weather-model run: run_hour_utc on the calendar day before the target day
run_time = pd.Timestamp(target_day.date()) - pd.Timedelta(days=1) + pd.Timedelta(hours=SETTINGS["run_hour_utc"])   # plain UTC, what Herbie wants
run_time_utc = run_time.tz_localize("UTC")

# forecast hours we need: from 1 hour after the run to the last hour of the target day
last_fxx = int((target_hours_utc[-1] - run_time_utc) / pd.Timedelta(hours=1))
fxx_list = list(range(1, last_fxx + 1))

print("Target day (Connecticut):  ", target_day.strftime("%A %B %d, %Y"))
print("Target hours in UTC:       ", target_hours_utc[0].strftime("%m-%d %H:%M"), "to", target_hours_utc[-1].strftime("%m-%d %H:%M"))
print("Weather model run (UTC):   ", run_time.strftime("%Y-%m-%d %H:%M"), "Z")
print("Same moment in Connecticut:", run_time_utc.tz_convert(TZ).strftime("%A %I:%M %p %Z"))
print("Forecast hours to download: f" + str(fxx_list[0]).zfill(3), "to f" + str(fxx_list[-1]).zfill(3), "(" + str(len(fxx_list)), "files per weather model)")

---
# Part 1. What actually happened

Before we can forecast anything, we need history: **how much electricity Connecticut used each hour**, and **what the temperature was each hour**.
The model learns the connection between the two from this history. The same data also gives us the actual answer for the target day.

## 1a. Electricity demand from ISO New England

**ISO New England (ISO-NE)** runs the power grid for the six New England states. Its public website, **ISO Express**, publishes a report called
**Five-Minute Estimated Zonal Load**: electricity demand every five minutes for each of the eight New England load zones. Connecticut is zone **4004**.

How the download works:

* The report is a CSV file at `https://www.iso-ne.com/transform/csv/fiveminuteestimatedzonalload?start=YYYYMMDD&end=YYYYMMDD`.
* ISO-NE only hands out the CSV if you visited the report's web page first. So we (1) open the web page once, which gives us a cookie,
  and (2) send that page's address along with every CSV request (the `Referer` header). A `requests.Session` keeps the cookie for us.
* We ask for **one day at a time**. A single day comes back in a fraction of a second. Asking for a whole month at once is slow and often gets cut off.

The file mixes comment lines (`"C"`), header lines (`"H"`) and data lines (`"D"`). Let's look at the first few lines of one day before writing the code that reads it.

In [ ]:
ISO_PAGE = "https://www.iso-ne.com/isoexpress/web/reports/load-and-demand/-/tree/estimated-zonal-load"
ISO_CSV = "https://www.iso-ne.com/transform/csv/fiveminuteestimatedzonalload"

iso = requests.Session()
iso.headers["User-Agent"] = "Mozilla/5.0 (educational notebook)"
first_visit = iso.get(ISO_PAGE, timeout=60)
print("visited the report page, status", first_visit.status_code)

day_text = target_day.strftime("%Y%m%d")
sample = iso.get(ISO_CSV + "?start=" + day_text + "&end=" + day_text, headers={"Referer": ISO_PAGE}, timeout=120)
lines = sample.text.splitlines()
print(len(lines), "lines for", target_day.date(), "\n")
for line in lines[:12]:
    print(line)

Each data line has: the time (Connecticut clock), the zone number, the zone name, the **estimated native load in MW**, and the estimated **behind-the-meter solar** in MW
(rooftop solar that the grid can't see directly). We keep Connecticut's rows and the load column.

The function below downloads one day and returns a small table. It tries up to three times, because any website hiccups now and then.

In [ ]:
def get_ct_load_one_day(day):
    """Five-minute Connecticut load for one day (Connecticut clock). Returns a table with time_local and load_mw."""
    day_text = day.strftime("%Y%m%d")
    url = ISO_CSV + "?start=" + day_text + "&end=" + day_text
    for attempt in range(3):
        try:
            response = iso.get(url, headers={"Referer": ISO_PAGE}, timeout=120)
            rows = []
            for line in response.text.splitlines():
                parts = line.replace('"', "").split(",")
                if len(parts) >= 6 and parts[0] == "D" and parts[2] == SETTINGS["load_zone_id"]:
                    rows.append({"time_local": parts[1], "load_mw": float(parts[4])})
            if len(rows) > 0:
                return pd.DataFrame(rows)
        except Exception as error:
            print("   ", day_text, "attempt", attempt + 1, "failed:", error)
        time.sleep(2)
    print("   ", day_text, "no data")
    return pd.DataFrame(columns=["time_local", "load_mw"])


one_day = get_ct_load_one_day(target_day)
print(len(one_day), "five-minute readings (288 in a normal day)")
one_day.head()

Now the whole history: one request per day, from `train_days` before the target day through the target day itself. About a year of days takes one to two minutes.

In [ ]:
first_day = (target_day.tz_localize(None) - pd.Timedelta(days=SETTINGS["train_days"] + 2)).normalize()
all_days = pd.date_range(first_day, target_day.tz_localize(None), freq="D")

pieces = []
start_clock = time.time()
for count, day in enumerate(all_days):
    pieces.append(get_ct_load_one_day(day))
    if count % 60 == 0:
        print("downloaded", count + 1, "of", len(all_days), "days")
five_minute = pd.concat(pieces, ignore_index=True)
print("done:", len(five_minute), "five-minute readings in", round(time.time() - start_clock), "seconds")

### From five-minute readings to hourly demand

* Convert the Connecticut clock time to UTC so it lines up with the weather data.
  (On the night the clocks fall back, 1:00 to 2:00 AM happens twice and the file can't tell the two apart, so we drop that one hour.)
* Average the twelve readings in each hour. The hour is labelled by its start: `14:00` means the average from 14:00 to 14:55.
* Keep hours with at least 10 of the 12 readings.

In [ ]:
times = pd.to_datetime(five_minute["time_local"], format="%m/%d/%Y %H:%M:%S")
five_minute["time_utc"] = times.dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")
five_minute = five_minute.dropna(subset=["time_utc"])
five_minute["hour_utc"] = five_minute["time_utc"].dt.floor("h")

hourly_groups = five_minute.groupby("hour_utc")["load_mw"]
load = hourly_groups.mean()
readings_per_hour = hourly_groups.size()
load = load[readings_per_hour >= 10]
load.name = "load_mw"
print(len(load), "hours of demand, from", load.index.min(), "to", load.index.max())
print("typical hour:", round(load.median()), "MW   lowest:", round(load.min()), "MW   highest:", round(load.max()), "MW")

In [ ]:
last_two_weeks = load[load.index >= load.index.max() - pd.Timedelta(days=14)]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(last_two_weeks.index.tz_convert(TZ), last_two_weeks.values, color="black")
ax.set_title("Connecticut electricity demand, last two weeks (hourly average, MW)", loc="left")
ax.set_ylabel("MW")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %m-%d", tz=TZ))
ax.grid(alpha=0.3)
plt.show()

You can see the daily rhythm: low overnight, a morning ramp as people wake up, a dip at midday (partly rooftop solar), and the evening peak around dinner time.

## 1b. Temperature at Bradley Airport

Airports record the weather every hour for pilots. Iowa State University's **Iowa Environmental Mesonet (IEM)** keeps a free archive of all of them.
One request gives us the whole year for one station.

* `station=BDL`: Bradley International Airport.
* `data=tmpf`: temperature in °F.
* `tz=Etc/UTC`: times in UTC.
* `report_type=3` and `4`: the routine hourly report plus any special reports (issued when the weather changes quickly).

Cleaning: the routine report comes out at 51 minutes past each hour (for example 13:51), so we round each report to the nearest hour, prefer the routine one, and fill any gap
of up to 3 hours by drawing a straight line between the neighbours.

In [ ]:
temp_start = first_day - pd.Timedelta(days=2)
temp_end = target_day.tz_localize(None) + pd.Timedelta(days=2)
iem_params = {
    "station": SETTINGS["station"],
    "data": "tmpf",
    "year1": temp_start.year, "month1": temp_start.month, "day1": temp_start.day,
    "year2": temp_end.year, "month2": temp_end.month, "day2": temp_end.day,
    "tz": "Etc/UTC",
    "format": "onlycomma",
    "report_type": [3, 4],
}
iem = requests.get("https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py", params=iem_params, timeout=180)
raw_weather = pd.read_csv(io.StringIO(iem.text), na_values=["M"])
print(len(raw_weather), "reports")
raw_weather.head()

In [ ]:
raw_weather["valid"] = pd.to_datetime(raw_weather["valid"], utc=True)
raw_weather["tmpf"] = pd.to_numeric(raw_weather["tmpf"], errors="coerce")
raw_weather = raw_weather.dropna(subset=["tmpf"])
raw_weather["hour_utc"] = raw_weather["valid"].dt.round("h")
raw_weather["is_routine"] = raw_weather["valid"].dt.minute == 51
raw_weather["minutes_off"] = (raw_weather["valid"] - raw_weather["hour_utc"]).abs().dt.total_seconds() / 60

# one report per hour: the routine one if there is one, otherwise the closest one
raw_weather = raw_weather.sort_values(["hour_utc", "is_routine", "minutes_off"], ascending=[True, False, True])
one_per_hour = raw_weather.drop_duplicates("hour_utc", keep="first").set_index("hour_utc")

hourly_grid = pd.date_range(one_per_hour.index.min(), one_per_hour.index.max(), freq="h")
observed_temp = one_per_hour["tmpf"].reindex(hourly_grid).interpolate(limit=3)
observed_temp.name = "temp_f"
print(len(observed_temp), "hours of temperature,", int(observed_temp.isna().sum()), "still missing")

In [ ]:
recent_temp = observed_temp[observed_temp.index >= load.index.max() - pd.Timedelta(days=14)]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(recent_temp.index.tz_convert(TZ), recent_temp.values, color="firebrick")
ax.set_title("Temperature at Bradley Airport, last two weeks (°F)", loc="left")
ax.set_ylabel("°F")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %m-%d", tz=TZ))
ax.grid(alpha=0.3)
plt.show()

---
# Part 2. The model

## 2a. Features: what the model gets to see

A model can't use a timestamp directly, so we turn each hour into numbers. Every feature below must be something we will **also know for tomorrow**,
because tomorrow's temperature will come from a weather forecast and the clock is always known.

| Feature | What it is | Why it helps |
|---|---|---|
| `hour` | hour of day, Connecticut time (0 to 23) | people's daily routine |
| `day_of_week` | 0 = Monday ... 6 = Sunday | weekdays use more power (offices, factories) |
| `is_holiday` | 1 on federal holidays | holidays look like Sundays |
| `day_of_year` | 1 to 366 | season: daylight hours, school calendar |
| `temp_f` | temperature this hour | heating and cooling |
| `temp_3h_ago` | temperature three hours earlier | buildings take a few hours to warm up or cool down |
| `temp_mean_24h` | average temperature over the last 24 hours | a cold snap that has lasted all day matters more than one cold hour |

**Training uses the measured temperature. Forecasting uses a weather model's temperature.** Same features, different source. That swap is the whole idea of this notebook.

The function takes a temperature series (one value per hour) and returns the feature table. We'll call it on measured temperatures now, and on each weather model later.

In [ ]:
holidays = USFederalHolidayCalendar().holidays(start="2015-01-01", end="2035-12-31")
FEATURES = ["hour", "day_of_week", "is_holiday", "day_of_year", "temp_f", "temp_3h_ago", "temp_mean_24h"]


def make_features(temps):
    """temps: hourly temperature (F) with a UTC time index and no missing hours in the grid. Returns one row of features per hour."""
    temps = temps.asfreq("h")
    local = temps.index.tz_convert(TZ)
    local_dates = local.tz_localize(None).normalize()
    table = pd.DataFrame(index=temps.index)
    table["hour"] = local.hour
    table["day_of_week"] = local.dayofweek
    table["is_holiday"] = local_dates.isin(holidays).astype(int)
    table["day_of_year"] = local.dayofyear
    table["temp_f"] = temps.values
    table["temp_3h_ago"] = temps.shift(3).values
    table["temp_mean_24h"] = temps.rolling(24).mean().values
    return table


history = make_features(observed_temp)
history["load_mw"] = load.reindex(history.index)
history = history.dropna()
print(history.shape[0], "hours with features and demand")
history.tail()

## 2b. Train the model, then check it

**Honesty rule:** the model may only learn from hours **before** the weather-model run. That's all a forecaster would have had at the time.

Before forecasting the target day, a quick check: hide the **last 14 days** before the run, train on everything older, and predict those 14 days.
This check uses the **measured** temperature, so it shows how good the model is when the weather is known perfectly. That's the best we can hope for with a weather forecast.

**The model: quantile random forest.** Like a random forest, it grows many decision trees that split the hours into groups (for example *"weekday, 5 PM, below 40 °F"*). A regular random forest would predict the *average* demand of the matching hours. A quantile forest keeps all the matching hours and reports their spread: the **10th percentile** (low), the **50th percentile** (middle, the median), and the **90th percentile** (high). If the model is honest, actual demand should land between low and high about 80% of the time.

Settings: 300 trees, and each final group must contain at least 5 hours (`min_samples_leaf=5`) so no tree memorises single hours.

In [ ]:
QUANTILES = [0.1, 0.5, 0.9]


def new_model():
    return RandomForestQuantileRegressor(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)


def predict_load(model, feature_table):
    """low (10th percentile), middle (median) and high (90th percentile) MW per hour"""
    values = model.predict(feature_table[FEATURES], quantiles=QUANTILES)
    return pd.DataFrame(values, index=feature_table.index, columns=["low", "middle", "high"])

In [ ]:
check_start = run_time_utc - pd.Timedelta(days=14)
train_part = history[history.index < check_start]
check_part = history[(history.index >= check_start) & (history.index < run_time_utc)]
print("training on", len(train_part), "hours, checking on", len(check_part), "hours")

check_model = new_model()
check_model.fit(train_part[FEATURES], train_part["load_mw"])
check_pred = predict_load(check_model, check_part)

In [ ]:
check_miss = (check_pred["middle"] - check_part["load_mw"]).abs().mean()
inside = ((check_part["load_mw"] >= check_pred["low"]) & (check_part["load_mw"] <= check_pred["high"])).mean()
print("average miss of the middle value over the 14 check days:", round(check_miss), "MW per hour (with perfect weather)")
print("actual demand landed between low and high in", round(100 * inside), "% of hours (honest would be about 80%)")

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.fill_between(check_pred.index.tz_convert(TZ), check_pred["low"], check_pred["high"], color="royalblue", alpha=0.2, label="low to high (80% range)")
ax.plot(check_part.index.tz_convert(TZ), check_part["load_mw"], color="black", linewidth=2, label="actual")
ax.plot(check_pred.index.tz_convert(TZ), check_pred["middle"], color="royalblue", label="middle (median)")
ax.set_title("Check: last 14 days before the forecast, using measured temperature", loc="left")
ax.set_ylabel("MW")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %m-%d", tz=TZ))
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Which features did the model lean on?

Feature importance measures how much each feature helped the trees split the hours into groups with similar demand. Bigger means more useful.

In [ ]:
importance = pd.Series(check_model.feature_importances_, index=FEATURES).sort_values()
importance.plot.barh(figsize=(8, 3.5), color="royalblue", title="Feature importance")
plt.show()

### The final model

Now train again on **every hour up to the weather-model run**, including those 14 days. This is the model we use for the target day.

In [ ]:
train_all = history[history.index <= run_time_utc]
model = new_model()
model.fit(train_all[FEATURES], train_all["load_mw"])
print("final model trained on", len(train_all), "hours, the last one", train_all.index.max().tz_convert(TZ).strftime("%a %b %d %I %p %Z"))

---
# Part 3. The weather forecasts

## How weather-model data is stored, and how we pull one number out of it

A weather model is a huge physics simulation run by a forecasting center (here, NOAA). Each **run** starts from the current state of the atmosphere
and steps forward in time. For every forecast hour it writes a file in **GRIB2** format, a compact binary format for gridded data.

* Each file covers the **whole map** (all of North America, or the whole globe) on a grid of points, and holds **hundreds of variables**:
  temperature at many heights, wind, humidity, cloud cover, rain, and so on. A single file can be 100 MB or more.
* Next to each file sits a small text **index file** (`.idx`) listing every variable and the **byte range** where it lives inside the big file.
* **Herbie** reads the index, finds the line for the variable we ask for, and downloads **only those bytes**, often well under 1 MB instead of 100+ MB.

We ask for **2 m temperature** (temperature at 2 metres above the ground, the standard "air temperature" at head height), using a **search string** that matches the
variable's line in the index, for example `":TMP:2 m above ground"`.

Then we take the value at the **grid point nearest to Bradley Airport**. No averaging of neighbouring grid points, and no filling in between hours:
all three models here write a file **every hour**, so every hour of the target day is a real model value.

Finally, the model temperatures are in **kelvin**. °F = K × 9/5 − 459.67.

### The three weather models

| | HRRR | GFS | NBM |
|---|---|---|---|
| Full name | High-Resolution Rapid Refresh | Global Forecast System | National Blend of Models |
| Made by | NOAA | NOAA | NOAA |
| What it is | detailed short-range model, US only | global model, the US workhorse | a statistical blend of many models (including HRRR, GFS, ECMWF, Canadian), corrected against past weather station readings |
| Grid spacing | 3 km | 0.25° (about 20 to 28 km here) | 2.5 km |
| New run | every hour | every 6 hours (00, 06, 12, 18Z) | every hour |
| How far ahead | 18 h (48 h for the 00, 06, 12, 18Z runs) | 16 days | 11 days |
| Time step | 1 h | 1 h for the first 5 days, then 3 h | 1 h for the first 36 h, then coarser |
| Herbie name / product | `hrrr` / `sfc` | `gfs` / `pgrb2.0p25` | `nbm` / `co` |

## Helper functions used for all three models

1. `download_variable`: download one variable with Herbie, trying again if NOAA's server drops the connection.
2. `nearest_grid_value`: find the grid point closest to Bradley and return its value.
3. `get_temperature_forecast`: loop over every forecast hour, download 2 m temperature, keep the Bradley value.
4. `temperature_for_model`: glue the measured temperatures (up to the run time) to the forecast temperatures (after the run time).
   At 2 PM the afternoon before, a forecaster knows the temperature so far and has to forecast the rest.
5. `forecast_demand`: build the features and run the trained model.

In [ ]:
def nearest_grid_value(dataset):
    """Value of the (only) variable in a Herbie/xarray dataset at the grid point nearest Bradley.
    Returns (value, grid latitude, grid longitude, distance in km)."""
    variable = list(dataset.data_vars)[0]
    lats = dataset["latitude"].values
    lons = dataset["longitude"].values
    if lats.ndim == 1:                      # GFS: a regular latitude/longitude grid stored as two 1-D lists
        lons, lats = np.meshgrid(lons, lats)
    lons = np.where(lons > 180, lons - 360, lons)     # some grids count longitude 0 to 360 instead of -180 to 180
    km_north = (lats - SETTINGS["latitude"]) * 111.0
    km_east = (lons - SETTINGS["longitude"]) * 111.0 * np.cos(np.radians(SETTINGS["latitude"]))
    distance = np.sqrt(km_north ** 2 + km_east ** 2)
    row, col = np.unravel_index(np.argmin(distance), distance.shape)
    value = float(np.asarray(dataset[variable].values)[row, col])
    return value, float(lats[row, col]), float(lons[row, col]), float(distance[row, col])


def download_variable(H, search):
    """H.xarray with patience: NOAA's servers sometimes drop a connection, so wait and try again (up to 6 times)."""
    for attempt in range(6):
        try:
            return H.xarray(search, remove_grib=True)
        except Exception as error:
            if attempt == 5:
                raise
            print("    connection problem, trying again in", 5 * (attempt + 1), "seconds:", str(error)[:80])
            time.sleep(5 * (attempt + 1))


def kelvin_to_f(kelvin):
    return kelvin * 9 / 5 - 459.67


def get_temperature_forecast(model_name, product, search_for, label):
    """Download 2 m temperature for every forecast hour of one run; return a Series of °F indexed by valid time (UTC)."""
    values = {}
    start_clock = time.time()
    for fxx in fxx_list:
        search = search_for(fxx)
        try:
            H = Herbie(run_time, model=model_name, product=product, fxx=fxx, priority=["aws"], verbose=False)
            dataset = download_variable(H, search)
            kelvin, grid_lat, grid_lon, km = nearest_grid_value(dataset)
            valid_time = run_time_utc + pd.Timedelta(hours=fxx)
            values[valid_time] = kelvin_to_f(kelvin)
        except Exception as error:
            print("   ", label, "f" + str(fxx).zfill(3), "gave up:", str(error)[:120])
        if fxx % 10 == 0:
            print("   ", label, ": forecast hour", fxx, "of", fxx_list[-1])
    series = pd.Series(values, name=label).sort_index()
    print(label, ":", len(series), "of", len(fxx_list), "hours downloaded in", round(time.time() - start_clock), "seconds")
    return series


def temperature_for_model(forecast_temps):
    """Measured temperature up to the run time, then the weather model's forecast."""
    hours = pd.date_range(target_hours_utc[0] - pd.Timedelta(hours=48), target_hours_utc[-1], freq="h")
    temps = observed_temp.reindex(hours)
    after_run = hours > run_time_utc
    temps[after_run] = forecast_temps.reindex(hours[after_run]).values
    missing = temps[after_run].isna()
    if missing.any():
        # a file that never downloaded: borrow the average of the hours on either side, and say so
        print("note:", int(missing.sum()), "forecast hour(s) missing, filled from the neighbouring hours:", list(temps[after_run][missing].index.strftime("%m-%d %H:%M UTC")))
        temps = temps.interpolate(limit=2)
    return temps


def forecast_demand(temps):
    """Features from a temperature series, then the model's demand forecast for the target day."""
    features = make_features(temps).loc[target_hours_utc]
    return predict_load(model, features)


actual_temp = observed_temp.reindex(target_hours_utc)
actual_load = load.reindex(target_hours_utc)
x_hours = target_hours_local
print("actual temperature known for", int(actual_temp.notna().sum()), "of", len(target_hours_utc), "target hours")
print("actual demand known for", int(actual_load.notna().sum()), "of", len(target_hours_utc), "target hours")

In [ ]:
def plot_model(label, color, forecast_temps, demand):
    """Two panels for one weather model: temperature forecast vs actual, demand forecast (with range) vs actual."""
    temp_target = forecast_temps.reindex(target_hours_utc)
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
    ax = axes[0]
    ax.plot(x_hours, actual_temp.values, color="black", linewidth=2.5, label="actual (Bradley)")
    ax.plot(x_hours, temp_target.values, color=color, marker="o", markersize=4, label=label + " forecast")
    ax.set_title(label + ": temperature (°F)", loc="left")
    ax = axes[1]
    ax.fill_between(x_hours, demand["low"].values, demand["high"].values, color=color, alpha=0.2, label="low to high (80% range)")
    ax.plot(x_hours, actual_load.values, color="black", linewidth=2.5, label="actual demand")
    ax.plot(x_hours, demand["middle"].values, color=color, marker="o", markersize=4, label="middle, from " + label)
    ax.set_title(label + ": Connecticut demand (MW)", loc="left")
    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%I %p", tz=TZ))
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(target_day.strftime("%A %B %d") + ", forecast made " + run_time_utc.tz_convert(TZ).strftime("%a %I %p %Z"), x=0.01, ha="left")
    plt.tight_layout()
    plt.show()
    temp_miss = (temp_target - actual_temp).abs().mean()
    temp_bias = (temp_target - actual_temp).mean()
    load_miss = (demand["middle"] - actual_load).abs().mean()
    inside = ((actual_load >= demand["low"]) & (actual_load <= demand["high"]))[actual_load.notna()].mean()
    print("temperature: average miss", round(temp_miss, 1), "°F, average error", round(temp_bias, 1), "°F (below 0 = forecast too cold)")
    print("demand: average miss of the middle value", round(load_miss), "MW per hour")
    print("actual demand inside the low-to-high range:", round(100 * inside), "% of hours")

## 3a. HRRR (High-Resolution Rapid Refresh)

The HRRR is NOAA's most detailed model for the United States. Its grid points are only **3 km** apart, so it can see things like the Connecticut River valley and
sea breezes off Long Island Sound. It starts a new run **every hour**, built on fresh radar and weather observations. Most runs look 18 hours ahead.
The 00, 06, 12 and 18Z runs go out to **48 hours**, which is why we use the 18Z run: it reaches the end of tomorrow.

**Where the files live:** NOAA's public bucket on Amazon Web Services, `noaa-hrrr-bdp-pds`. The product `sfc` is the "surface" file
(variables near the ground: 2 m temperature, 10 m wind, and so on).

### Step 1: look at one file before downloading all of them

`Herbie(...)` doesn't download anything yet. It finds where the file is. `priority=["aws"]` tells it to use NOAA's archive on Amazon's cloud, which keeps every run (NOAA's own NOMADS server only keeps the last couple of days). `H.inventory(search)` reads the index file and shows which line(s) match our search string,
including the byte range Herbie will download.

In [ ]:
def hrrr_search(fxx):
    return ":TMP:2 m above ground"


sample_fxx = fxx_list[-1]   # one file to look at: the last hour of the target day
H = Herbie(run_time, model="hrrr", product="sfc", fxx=sample_fxx, priority=["aws"])
print(H)
H.inventory(hrrr_search(sample_fxx))

### Step 2: download that one variable and look inside

`H.xarray(search)` (wrapped in our `download_variable`, which retries) downloads just the matching bytes and opens them as an **xarray Dataset**: a grid of numbers with latitude and longitude for every point.
The temperature variable is called `t2m`, in kelvin. Then `nearest_grid_value` picks out the point closest to Bradley.

In [ ]:
one_file = download_variable(H, hrrr_search(sample_fxx))
print(one_file)
kelvin, grid_lat, grid_lon, km = nearest_grid_value(one_file)
print()
print("grid size:", one_file["t2m"].shape)
print("nearest grid point:", round(grid_lat, 3), round(grid_lon, 3), "which is", round(km, 1), "km from Bradley")
print("HRRR temperature there, valid", (run_time_utc + pd.Timedelta(hours=sample_fxx)).tz_convert(TZ).strftime("%a %I %p %Z"), ":", round(kelvin, 2), "K =", round(kelvin_to_f(kelvin), 1), "°F")

### Step 3: every forecast hour

Same thing in a loop, one file per forecast hour, from f001 to the end of the target day. This is the slow part (roughly 1 to 3 minutes).

In [ ]:
hrrr_temps = get_temperature_forecast("hrrr", "sfc", hrrr_search, "HRRR")
hrrr_temps.head()

### Step 4: forecast demand from the HRRR temperatures, and compare with what happened

Left: HRRR's temperature forecast against the thermometer at Bradley. Right: the demand our model predicts from those temperatures, against actual Connecticut demand.

In [ ]:
hrrr_demand = forecast_demand(temperature_for_model(hrrr_temps))
plot_model("HRRR", "darkorange", hrrr_temps, hrrr_demand)

## 3b. GFS (Global Forecast System)

The GFS is NOAA's **global** model, the backbone of US weather forecasting. It covers the whole planet on a **0.25°** grid
(about 28 km north–south and 20 km east–west at Connecticut's latitude), so one grid box is roughly the size of Hartford County.
It runs **four times a day** (00, 06, 12, 18Z) and looks **16 days** ahead, with a file every hour for the first five days.

Coarser grid boxes average over more ground, so the GFS can miss local details: river valleys, the coast, or a cold pocket on a calm, clear night.

**Where the files live:** `noaa-gfs-bdp-pds` on AWS. The product `pgrb2.0p25` is the main 0.25° file.

In [ ]:
def gfs_search(fxx):
    return ":TMP:2 m above ground"


sample_fxx = fxx_list[-1]   # one file to look at: the last hour of the target day
H = Herbie(run_time, model="gfs", product="pgrb2.0p25", fxx=sample_fxx, priority=["aws"])
print(H)
H.inventory(gfs_search(sample_fxx))

### Step 2: download that one variable and look inside

`H.xarray(search)` (wrapped in our `download_variable`, which retries) downloads just the matching bytes and opens them as an **xarray Dataset**: a grid of numbers with latitude and longitude for every point.
The temperature variable is called `t2m`, in kelvin. Then `nearest_grid_value` picks out the point closest to Bradley.

In [ ]:
one_file = download_variable(H, gfs_search(sample_fxx))
print(one_file)
kelvin, grid_lat, grid_lon, km = nearest_grid_value(one_file)
print()
print("grid size:", one_file["t2m"].shape)
print("nearest grid point:", round(grid_lat, 3), round(grid_lon, 3), "which is", round(km, 1), "km from Bradley")
print("GFS temperature there, valid", (run_time_utc + pd.Timedelta(hours=sample_fxx)).tz_convert(TZ).strftime("%a %I %p %Z"), ":", round(kelvin, 2), "K =", round(kelvin_to_f(kelvin), 1), "°F")

### Step 3: every forecast hour

Same thing in a loop, one file per forecast hour, from f001 to the end of the target day. This is the slow part (roughly 1 to 3 minutes).

In [ ]:
gfs_temps = get_temperature_forecast("gfs", "pgrb2.0p25", gfs_search, "GFS")
gfs_temps.head()

### Step 4: forecast demand from the GFS temperatures, and compare with what happened

Left: GFS's temperature forecast against the thermometer at Bradley. Right: the demand our model predicts from those temperatures, against actual Connecticut demand.

In [ ]:
gfs_demand = forecast_demand(temperature_for_model(gfs_temps))
plot_model("GFS", "purple", gfs_temps, gfs_demand)

## 3c. NBM (National Blend of Models)

The NBM isn't a physics model of its own. It is NOAA's **blend**: it takes many models (HRRR, GFS, ECMWF, Canadian, ensembles and more), weights them by how well
each has done recently, and **corrects them against years of weather-station readings**. It's what National Weather Service forecasters start from,
so it's the closest thing here to "the official forecast". The grid is **2.5 km**, it runs **every hour**, and it has a file every hour for the first 36 hours.

**Where the files live:** `noaa-nbm-grib2-pds` on AWS. The product `co` is the CONUS (lower 48 states) file.

**One wrinkle:** the NBM file holds several temperature lines for the same hour (the blend itself plus, for example, a measure of how much the models disagree).
So our search string is stricter: it must say exactly `":TMP:2 m above ground:<hour> hour fcst:"` with nothing after it. The `$` at the end means "the line ends here".

In [ ]:
def nbm_search(fxx):
    return ":TMP:2 m above ground:" + str(fxx) + " hour fcst:$"


sample_fxx = fxx_list[-1]   # one file to look at: the last hour of the target day
H = Herbie(run_time, model="nbm", product="co", fxx=sample_fxx, priority=["aws"])
print(H)
H.inventory(nbm_search(sample_fxx))

### Step 2: download that one variable and look inside

`H.xarray(search)` (wrapped in our `download_variable`, which retries) downloads just the matching bytes and opens them as an **xarray Dataset**: a grid of numbers with latitude and longitude for every point.
The temperature variable is called `t2m`, in kelvin. Then `nearest_grid_value` picks out the point closest to Bradley.

In [ ]:
one_file = download_variable(H, nbm_search(sample_fxx))
print(one_file)
kelvin, grid_lat, grid_lon, km = nearest_grid_value(one_file)
print()
print("grid size:", one_file["t2m"].shape)
print("nearest grid point:", round(grid_lat, 3), round(grid_lon, 3), "which is", round(km, 1), "km from Bradley")
print("NBM temperature there, valid", (run_time_utc + pd.Timedelta(hours=sample_fxx)).tz_convert(TZ).strftime("%a %I %p %Z"), ":", round(kelvin, 2), "K =", round(kelvin_to_f(kelvin), 1), "°F")

### Step 3: every forecast hour

Same thing in a loop, one file per forecast hour, from f001 to the end of the target day. This is the slow part (roughly 1 to 3 minutes).

In [ ]:
nbm_temps = get_temperature_forecast("nbm", "co", nbm_search, "NBM")
nbm_temps.head()

### Step 4: forecast demand from the NBM temperatures, and compare with what happened

Left: NBM's temperature forecast against the thermometer at Bradley. Right: the demand our model predicts from those temperatures, against actual Connecticut demand.

In [ ]:
nbm_demand = forecast_demand(temperature_for_model(nbm_temps))
plot_model("NBM", "seagreen", nbm_temps, nbm_demand)

---
# Part 4. All three side by side

One more forecast for reference: **perfect weather**. We feed the model the temperature that *actually* happened. No weather forecast can do better than this,
so the gap between a weather model's line and the perfect-weather line is the part of the error caused by the **weather forecast**.
The gap between the perfect-weather line and the actual demand is the part caused by **our demand model**.

In [ ]:
perfect_demand = forecast_demand(observed_temp)

forecasts = {
    "HRRR": (hrrr_temps, hrrr_demand, "darkorange"),
    "GFS": (gfs_temps, gfs_demand, "purple"),
    "NBM": (nbm_temps, nbm_demand, "seagreen"),
}

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
ax = axes[0]
ax.plot(x_hours, actual_temp.values, color="black", linewidth=3, label="actual (Bradley)")
for label in forecasts:
    temps, demand, color = forecasts[label]
    ax.plot(x_hours, temps.reindex(target_hours_utc).values, color=color, marker="o", markersize=4, label=label)
ax.set_ylabel("°F")
ax.set_title("Temperature: three weather models vs the thermometer", loc="left")
ax = axes[1]
ax.plot(x_hours, actual_load.values, color="black", linewidth=3, label="actual demand")
ax.plot(x_hours, perfect_demand["middle"].values, color="grey", linestyle="--", linewidth=2, label="perfect weather (actual temperature)")
for label in forecasts:
    temps, demand, color = forecasts[label]
    ax.plot(x_hours, demand["middle"].values, color=color, marker="o", markersize=4, label="from " + label)
ax.set_ylabel("MW")
ax.set_title("Connecticut demand: our model fed each weather model", loc="left")
for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%I %p", tz=TZ))
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(target_day.strftime("%A %B %d, %Y") + ": forecasts made " + run_time_utc.tz_convert(TZ).strftime("%a %I %p %Z") + " the day before", x=0.01, ha="left")
plt.tight_layout()
plt.show()

### The scoreboard

* **temperature miss**: average distance between forecast and actual temperature, in °F.
* **temperature error**: the same but keeping the sign. Below 0 means the forecast was too cold on average.
* **demand miss**: average distance between predicted and actual demand, in MW per hour (this is the MAE, mean absolute error).
* **inside range**: share of hours where actual demand fell between the low and high values. About 80% means the range is honest.

In [ ]:
rows = []
for label in forecasts:
    temps, demand, color = forecasts[label]
    temp_target = temps.reindex(target_hours_utc)
    row = {
        "weather source": label,
        "temperature miss (°F)": (temp_target - actual_temp).abs().mean(),
        "temperature error (°F)": (temp_target - actual_temp).mean(),
        "demand miss (MW)": (demand["middle"] - actual_load).abs().mean(),
    }
    row["inside range (%)"] = 100 * ((actual_load >= demand["low"]) & (actual_load <= demand["high"]))[actual_load.notna()].mean()
    rows.append(row)
perfect_row = {
    "weather source": "perfect weather",
    "temperature miss (°F)": 0.0,
    "temperature error (°F)": 0.0,
    "demand miss (MW)": (perfect_demand["middle"] - actual_load).abs().mean(),
}
perfect_row["inside range (%)"] = 100 * ((actual_load >= perfect_demand["low"]) & (actual_load <= perfect_demand["high"]))[actual_load.notna()].mean()
rows.append(perfect_row)
scoreboard = pd.DataFrame(rows).set_index("weather source").round(1)
scoreboard

### How to read this

* If a weather model's **temperature miss** is small but its **demand miss** is still big, and close to the perfect-weather row, the weather was fine and the demand model is the weak link.
* If the perfect-weather row is much better than the weather models, the **weather forecast** is the weak link that day.
* One day is one day. A model that wins today can lose tomorrow. Real comparisons use weeks or months of forecasts.

### Things to try

1. Change `target_day` to a hot summer day or a cold winter day. Which model does best when the weather is extreme?
2. Change `run_hour_utc` to `0` (the run from the night before, 8 PM in Connecticut) or `12` (that morning, 8 AM), and see whether newer runs help.
3. Add a feature. Ideas: cloud cover (the HRRR, GFS and NBM files all have it, search for `":TCDC:"`), humidity, or wind.
4. Use more training history with `train_days`.
5. Average the three models' temperatures into one "blend" and see whether it beats each one alone.